### 生成 Node2Vec 嵌入

In [5]:
import torch
from torch_geometric.nn import Node2Vec
import pandas as pd
import numpy as np
import pickle
import os
import gc  # 引入垃圾回收模块
import warnings

# 忽略警告
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

# ================= 配置 =================
DATA_PATH = '../data/kg.csv'
OUTPUT_NPY = '../data/node2vec_embeddings.npy'
OUTPUT_MAP = '../data/node2vec_id_map.pkl'

EMBEDDING_DIM = 128
WALK_LENGTH = 30
CONTEXT_SIZE = 10
WALKS_PER_NODE = 10
EPOCHS = 50
BATCH_SIZE = 256

# =====================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f">> 使用设备: {device}")

# 1. 加载数据
print(">> 正在加载数据...")
df = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)
print(f"✅ 数据加载完成，共 {len(df)} 条边。")

# 2. 构建图索引
node_to_id = {}
id_to_node = []
edge_index_list = [[], []]

def get_id(node_str):
    if node_str not in node_to_id:
        node_to_id[node_str] = len(node_to_id)
        id_to_node.append(node_str)
    return node_to_id[node_str]

print(">> 正在构建图索引 (这可能有点慢，请耐心等待)...")
# 简单的迭代，减少开销
for _, row in df.iterrows():
    src = f"{row['x_type']}::{row['x_index']}"
    dst = f"{row['y_type']}::{row['y_index']}"
    
    u = get_id(src)
    v = get_id(dst)
    
    edge_index_list[0].append(u)
    edge_index_list[1].append(v)

edge_index = torch.tensor(edge_index_list, dtype=torch.long)
num_nodes = len(node_to_id)

print(f"\n✅ 图构建完成！节点数: {num_nodes}, 边数: {edge_index.shape[1]}")

# 🚀【关键优化】手动释放 DataFrame 和列表的内存，为模型训练腾出空间
del df
del edge_index_list
gc.collect() 
print(">> 已清理原始数据内存，准备初始化模型...")

# 3. 初始化模型
model = Node2Vec(
    edge_index=edge_index,
    embedding_dim=EMBEDDING_DIM,
    walk_length=WALK_LENGTH,
    context_size=CONTEXT_SIZE,
    walks_per_node=WALKS_PER_NODE,
    sparse=True
).to(device)

# 释放 edge_index (模型内部已经持有了引用，或者我们可以保留它，视 PyG 版本而定)
# 为了安全，我们保留 edge_index 直到 loader 创建完成
loader = model.loader(batch_size=BATCH_SIZE)
del edge_index  # loader 创建后可以删除原始的 edge_index 张量
gc.collect()

optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

# 4. 训练
print(">> 开始训练 Node2Vec...")
model.train()

for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    # 注意：如果 NUM_WORKERS > 0，loader 是多进程的，这里不会阻塞太多内存
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss / len(loader):.4f}")

print("✅ 训练完成！正在保存结果...")

# 5. 提取并保存 (优化版：拆分为 .npy 和 .pkl)
model.eval()
with torch.no_grad():
    # 获取 CPU 上的 numpy 数组
    embedding_matrix = model.embedding.weight.cpu().numpy()

# 保存映射表 (只存字符串列表，很小)
with open(OUTPUT_MAP, 'wb') as f:
    pickle.dump(id_to_node, f, protocol=pickle.HIGHEST_PROTOCOL)

# 保存矩阵 (二进制，极快且小)
np.save(OUTPUT_NPY, embedding_matrix)

# 计算文件大小
size_npy = os.path.getsize(OUTPUT_NPY) / 1024 / 1024
size_map = os.path.getsize(OUTPUT_MAP) / 1024 / 1024
print(f"\n✅ 成功保存！")
print(f"   矩阵文件 (.npy): {size_npy:.2f} MB")
print(f"   映射文件 (.pkl): {size_map:.2f} MB")
print(f"   总计: {size_npy + size_map:.2f} MB (远小于之前的 8GB+)")

# 🚀【关键优化】彻底清理模型，防止后续代码爆内存
del model
del embedding_matrix
del id_to_node
del node_to_id
gc.collect()

print(">> 内存已清理，脚本安全结束。你可以安全地加载 .npy 和 .pkl 文件进行下一步了。")

>> 使用设备: cuda
>> 正在加载数据...
✅ 数据加载完成，共 8100498 条边。
>> 正在构建图索引 (这可能有点慢，请耐心等待)...

✅ 图构建完成！节点数: 129375, 边数: 8100498
>> 已清理原始数据内存，准备初始化模型...
>> 开始训练 Node2Vec...
Epoch 10, Loss: 0.8487
Epoch 20, Loss: 0.8405
Epoch 30, Loss: 0.8396
Epoch 40, Loss: 0.8392
Epoch 50, Loss: 0.8392
✅ 训练完成！正在保存结果...

✅ 成功保存！
   矩阵文件 (.npy): 63.17 MB
   映射文件 (.pkl): 2.82 MB
   总计: 65.99 MB (远小于之前的 8GB+)
>> 内存已清理，脚本安全结束。你可以安全地加载 .npy 和 .pkl 文件进行下一步了。


###生成 PubMedBERT 嵌入

In [4]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import pandas as pd
import torch
import numpy as np
import pickle

import gc
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm



# ================= 配置 =================
DATA_PATH = '../data/kg.csv'
OUTPUT_NPY = '../data/pubmedbert_embeddings.npy'
OUTPUT_MAP = '../data/pubmedbert_id_map.pkl'

BATCH_SIZE = 32  # 如果显存不够，改小为 16 或 8
MAX_LENGTH = 512
MODEL_NAME = "microsoft/pubmedbert-base-uncased"
# =====================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f">> 使用设备: {device}")

# 1. 加载数据
print(">> 正在加载数据...")
df = pd.read_csv(DATA_PATH, dtype=str, low_memory=False)

# 2. 构建唯一节点列表和对应的文本
nodes_map = {} 

print(">> 正在提取唯一节点及其名称...")
for _, row in df.iterrows():
    # 处理源节点
    src_id = f"{row['x_type']}::{row['x_index']}"
    src_name = str(row['x_name']).strip() if pd.notna(row['x_name']) and str(row['x_name']).strip() != "" else f"{row['x_type']} {row['x_index']}"
    
    if src_id not in nodes_map:
        nodes_map[src_id] = src_name
    
    # 处理目标节点
    dst_id = f"{row['y_type']}::{row['y_index']}"
    dst_name = str(row['y_name']).strip() if pd.notna(row['y_name']) and str(row['y_name']).strip() != "" else f"{row['y_type']} {row['y_index']}"
    
    if dst_id not in nodes_map:
        nodes_map[dst_id] = dst_name

unique_ids = list(nodes_map.keys())
unique_texts = [nodes_map[i] for i in unique_ids]

num_nodes = len(unique_ids)
print(f"✅ 共提取 {num_nodes} 个唯一节点。")

# 释放原始 DataFrame 内存
del df
del nodes_map
gc.collect()


>> 使用设备: cuda
>> 正在加载数据...
>> 正在提取唯一节点及其名称...
✅ 共提取 129375 个唯一节点。


2518

In [5]:

# 3. 加载模型
print(f">> 正在加载模型 {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

# 4. 批量推理
print(">> 开始生成嵌入 (这可能花费几分钟)...")
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, num_nodes, BATCH_SIZE), desc="Encoding"):
        batch_texts = unique_texts[i : i + BATCH_SIZE]
        
        inputs = tokenizer(
            batch_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=MAX_LENGTH
        ).to(device)
        
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        
        all_embeddings.append(cls_embeddings)
        
        # 清理显存
        del inputs
        del outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# 5. 合并并保存
print(">> 正在合并结果并保存...")
final_matrix = np.concatenate(all_embeddings, axis=0)

# 保存映射表
with open(OUTPUT_MAP, 'wb') as f:
    pickle.dump(unique_ids, f, protocol=pickle.HIGHEST_PROTOCOL)

# 保存矩阵
np.save(OUTPUT_NPY, final_matrix)

print(f"\n✅ PubMedBERT 嵌入生成完成！")
print(f"   矩阵维度: {final_matrix.shape}")  # 这里现在可以正常显示了
size_npy = os.path.getsize(OUTPUT_NPY) / 1024 / 1024
size_map = os.path.getsize(OUTPUT_MAP) / 1024 / 1024
print(f"   矩阵文件 (.npy): {size_npy:.2f} MB")
print(f"   映射文件 (.pkl): {size_map:.2f} MB")

# 最后再清理内存
del all_embeddings
del final_matrix
del unique_ids
del model
del tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(">> 内存已清理，脚本安全结束。")

>> 正在加载模型 microsoft/pubmedbert-base-uncased ...


'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/microsoft/pubmedbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/microsoft/pubmedbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'microsoft/pubmedbert-base-uncased'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'microsoft/pubmedbert-base-uncased' is the correct path to a directory containing a config.json file